In [2]:
import pandas as pd
import numpy as np

# --- Phase 1: Data Loading and Initial Inspection ---

# Load dataset
df = pd.read_csv("C:\\Users\\Shagun Jain\\OneDrive\\Documents\\online retail csv.csv")

# Display first few rows
print("First 5 rows of the dataset:")
print(df.head())

# Print data types
print("\nData types of each column:")
print(df.dtypes)

# Number of rows and columns
print("\nDataset shape:", df.shape)

# Initial Data Exploration
print("\nMissing values per column:")
print(df.isnull().sum())

# Descriptive statistics for numerical columns
print("\nDescriptive statistics:")
print(df.describe())

# Identify potential outliers in Quantity and UnitPrice
for col in ['Quantity', 'UnitPrice']:
    mean = df[col].mean()
    std = df[col].std()
    outliers = df[(df[col] > mean + 3 * std) | (df[col] < mean - 3 * std)][col]
    print(f"\nPotential outliers in {col}:", outliers.count())

# Observations
print("\nInitial Observations: Check for negative Quantity, missing CustomerID, and inconsistent Description.")

# --- Phase 2: Data Cleaning ---

# 1. Handling Missing Values
# Drop rows with missing UnitPrice (critical for analysis)
df['UnitPrice'] = pd.to_numeric(df['UnitPrice'], errors='coerce')  # Ensure numeric
df.dropna(subset=['UnitPrice'], inplace=True)

# For CustomerID, assign a placeholder (-1) for missing values
df['CustomerID'] = df['CustomerID'].fillna(-1)  # Placeholder for unidentified customers
print("\nMissing values after handling:")
print(df.isnull().sum())

# Justification: UnitPrice is essential; missing CustomerID can be tagged for analysis.

# 2. Data Type Conversion
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')  # Convert to datetime
df['CustomerID'] = df['CustomerID'].astype(int)  # Convert to integer
df['Country'] = df['Country'].astype('category')  # Convert to categorical
print("\nUpdated data types:")
print(df.dtypes)

# 3. Handling Duplicates
# Remove duplicate rows based on InvoiceNo and CustomerID
df = df.drop_duplicates(subset=['InvoiceNo', 'CustomerID'], keep='first')
print("\nShape after removing duplicates:", df.shape)

# 4. Addressing Inconsistencies
# Standardize Country names (title case)
df['Country'] = df['Country'].str.title()

# Clean Description (remove extra spaces, uppercase)
df['Description'] = df['Description'].str.strip().str.upper()
print("\nUnique countries after standardization:", df['Country'].unique())

# 5. Outlier Handling
# Cap Quantity and UnitPrice at 99th percentile
for col in ['Quantity', 'UnitPrice']:
    cap_value = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=cap_value)
print("\nMax Quantity after capping:", df['Quantity'].max())
print("Max UnitPrice after capping:", df['UnitPrice'].max())

# Justification: Capping preserves data while mitigating extreme outliers.

# --- Phase 3: Data Transformation ---

# 1. Feature Engineering
# Calculate total purchase value (Quantity * UnitPrice)
df['TotalPurchase'] = df['Quantity'] * df['UnitPrice']

# Extract month from InvoiceDate
df['PurchaseMonth'] = df['InvoiceDate'].dt.month
print("\nSample of new features:")
print(df[['TotalPurchase', 'PurchaseMonth']].head())

# Rationale: TotalPurchase reflects customer spending; PurchaseMonth aids temporal analysis.

# 2. Data Aggregation and Summarization
# Average TotalPurchase per Country
country_summary = df.groupby('Country')['TotalPurchase'].agg(['mean', 'count'])
print("\nCountry summary:")
print(country_summary)

# Pivot table: TotalPurchase by Country and PurchaseMonth
pivot = df.pivot_table(values='TotalPurchase', index='Country', columns='PurchaseMonth', aggfunc='mean')
print("\nPivot table (TotalPurchase by Country and Month):")
print(pivot)

# 3. Data Standardization/Normalization
# Normalize TotalPurchase (min-max scaling)
df['TotalPurchase_normalized'] = (df['TotalPurchase'] - df['TotalPurchase'].min()) / \
                                 (df['TotalPurchase'].max() - df['TotalPurchase'].min())
print("\nNormalized TotalPurchase sample:")
print(df['TotalPurchase_normalized'].head())

# When needed: For models requiring comparable scales.

# 4. Data Binning
# Bin Quantity into categories
quantity_bins = [0, 10, 50, 100, float('inf')]
quantity_labels = ['Low', 'Medium', 'High', 'Very High']
df['QuantityCategory'] = pd.cut(df['Quantity'], bins=quantity_bins, labels=quantity_labels, right=False)
print("\nQuantity category distribution:")
print(df['QuantityCategory'].value_counts())

# --- Phase 4: Reporting and Documentation ---

# 1. Save Cleaned Dataset
df.to_csv(r"C:\Users\Shagun Jain\OneDrive\Documents\cleaned_online_retail.csv", index=False)
print("\nCleaned dataset saved as 'cleaned_online_retail.csv'")

# 2. Data Wrangling Report (Printed Summary)
print("\n--- Data Wrangling Report ---")
print("Initial Findings: Missing CustomerID (25% rows), negative Quantity values, inconsistent Description.")
print("Cleaning Decisions: Dropped missing UnitPrice rows; tagged missing CustomerID as -1; capped outliers.")
print("Transformations: Added TotalPurchase and PurchaseMonth; binned Quantity for segmentation.")
print("Assumptions: Negative Quantity treated as returns, capped at 0 if needed; missing CustomerID valid for analysis.")
print("Summary: Final dataset has", df.shape[0], "rows, no missing UnitPrice, and enhanced features.")

First 5 rows of the dataset:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

        InvoiceDate  UnitPrice  CustomerID         Country  
0  01-12-2010 08:26       2.55     17850.0  United Kingdom  
1  01-12-2010 08:26       3.39     17850.0  United Kingdom  
2  01-12-2010 08:26       2.75     17850.0  United Kingdom  
3  01-12-2010 08:26       3.39     17850.0  United Kingdom  
4  01-12-2010 08:26       3.39     17850.0  United Kingdom  

Data types of each column:
InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
Custom